# Data Quality Audit — `b2b_ecommerce.audit.data_quality_audit`

**Row-level exceptions**: one row per failed record × failed rule, across all 9 silver tables of
`bonze_to_silver_etl`. No aggregates — every exception points at a specific record.

**Where exceptions come from:**

* **Drop-level rejects** — read from the quarantine tables (`b2b_ecommerce.quarantine.leads_silver`,
  `logs_silver`): each quarantined row × each rule in its `failed_checks`.
* **Warn-level violations** — re-evaluated against the **current** silver table (the row stays in
  silver, the exception is only recorded). Silver is SCD Type 1 everywhere (upserts, no
  version columns — gold owns the SCD2 history), so every row is current.
* **Referential warn rules** — evaluated via anti-join against the current reference table.

**rule_type legend:** `NOT_NULL | FORMAT | RANGE | ENUM | REFERENTIAL | DATE_LOGIC | PARSE`
**exception_action legend:** `WARN_ROW_KEPT` (row is in silver) | `DROP_QUARANTINED` (row is in the
quarantine table)

The `WARN_RULES` / `QUAR_RULES` config below mirrors the expectations defined in
`bonze_to_silver/src/bonze_to_silver_etl/transformations/*` — keep it in sync when a rule changes there.

**History:** the pre-2026-11-16 aggregate-snapshot version of this audit is preserved as
`audit.data_quality_audit_legacy`. Bronze-layer drops (none today) are not row-capturable — the raw
source files remain the bronze safety net.


In [0]:
# Row-level DQ exception engine - config + run_dq_audit().
# Writes b2b_ecommerce.audit.data_quality_audit: ONE ROW PER FAILED
# RECORD x FAILED RULE. Three sources:
#   1) quarantine tables -> drop-level rejects (exception_action=DROP_QUARANTINED)
#   2) warn rules re-run against current silver -> WARN_ROW_KEPT
#   3) referential warn rules -> anti-join vs the current reference table
# The rule config below mirrors the expectations in
# bonze_to_silver/src/bonze_to_silver_etl/transformations/* - keep in sync.
import hashlib, uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

DQ_TABLE = "b2b_ecommerce.audit.data_quality_audit"
SILVER = "b2b_ecommerce.silver"
SILVER_PIPELINE = "bonze_to_silver_etl"

# silver is SCD Type 1 everywhere now (user redesign: upserts, no
# __END_AT version columns; gold owns the SCD2 history) - no table
# needs a current-version filter
SCD2 = set()

KEYS = {
    "leads": ["lead_id"], "logs": ["log_id"], "companies": ["company_id"],
    "customers": ["customer_id"], "orders": ["order_id"], "order_items": ["order_item_id"],
    "products": ["product_id"], "suppliers": ["supplier_id"],
    "supplier_products": ["supplier_id", "product_id"],
}

# (rule, rule_type, failed_column, predicate on current silver, reject_reason)
WARN_RULES = {
    "leads": [
        ("contact_email_valid", "FORMAT", "contact_email",
         "contact_email IS NULL OR contact_email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'",
         "contact_email is not a valid email address"),
        ("phone_format_valid", "FORMAT", "phone",
         "phone IS NULL OR phone RLIKE '^[+]?[0-9 ()-]+$'",
         "phone is not in a valid phone format"),
        ("company_name_not_null", "NOT_NULL", "company_name",
         "company_name IS NOT NULL", "company_name is missing"),
        ("contact_name_not_null", "NOT_NULL", "contact_name",
         "contact_name IS NOT NULL", "contact_name is missing"),
    ],
    "orders": [
        ("status_valid", "ENUM", "status",
         "status IN ('pending', 'confirmed', 'shipped', 'delivered', 'cancelled')",
         "status is outside the canonical values"),
        ("payment_method_valid", "ENUM", "payment_method",
         "payment_method IN ('credit_card', 'bank_transfer', 'cheque', 'cash_on_delivery')",
         "payment_method is outside the canonical values"),
        ("shipping_method_valid", "ENUM", "shipping_method",
         "shipping_method IN ('standard', 'express', 'pickup')",
         "shipping_method is outside the canonical values"),
        ("channel_valid", "ENUM", "channel",
         "channel IN ('web', 'email', 'phone', 'sales_rep')",
         "channel is outside the canonical values"),
        ("po_number_format_valid", "FORMAT", "po_number",
         "po_number IS NULL OR po_number RLIKE '^PO-[0-9]+$'",
         "po_number is not in PO-<digits> format"),
        ("order_date_valid", "DATE_LOGIC", "order_date",
         "order_date IS NOT NULL AND order_date <= CURRENT_DATE()",
         "order_date is missing or in the future - untrustworthy for time-based reporting"),
    ],
    "order_items": [
        ("warehouse_valid", "ENUM", "warehouse",
         "warehouse IN ('BsAs-Central', 'Rosario-Sur', 'Cordoba-Norte')",
         "warehouse is outside the canonical values"),
        ("discount_pct_valid", "RANGE", "discount_pct",
         "discount_pct >= 0 AND discount_pct <= 100",
         "discount_pct is outside 0-100"),
        ("tax_rate_valid", "RANGE", "tax_rate",
         "tax_rate >= 0", "tax_rate is negative"),
        ("quantity_valid", "RANGE", "quantity",
         "quantity > 0",
         "quantity is missing or non-positive (value NULLified in silver; line_total keeps the true extension)"),
        ("unit_price_valid", "RANGE", "unit_price",
         "unit_price > 0",
         "unit_price is missing or non-positive (value NULLified in silver; line_total keeps the true extension)"),
    ],
    "customers": [
        ("company_id_not_null", "NOT_NULL", "company_id",
         "company_id IS NOT NULL", "company_id is missing"),
        ("email_valid", "FORMAT", "email",
         "email IS NOT NULL AND email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'",
         "email is missing or invalid"),
        ("birth_date_not_future", "DATE_LOGIC", "birth_date",
         "birth_date IS NULL OR birth_date < CURRENT_DATE()",
         "birth_date is today or in the future"),
        ("phone_format_valid", "FORMAT", "phone",
         "phone IS NULL OR phone RLIKE '^[+][0-9]'",
         "phone is not in international +NN format"),
    ],
    "companies": [
        ("name_not_null", "NOT_NULL", "name",
         "name IS NOT NULL", "name is missing"),
        ("company_type_valid", "ENUM", "company_type",
         "company_type IN ('buyer', 'supplier')",
         "company_type is outside buyer/supplier"),
        ("contact_email_valid", "FORMAT", "contact_email",
         "contact_email IS NOT NULL AND contact_email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'",
         "contact_email is missing or invalid"),
        ("phone_format_valid", "FORMAT", "phone",
         "phone IS NULL OR phone RLIKE '^[+][0-9]'",
         "phone is not in international +NN format"),
        ("rating_range_valid", "RANGE", "rating",
         "rating IS NULL OR rating BETWEEN 0 AND 5",
         "rating is outside 0-5"),
        ("employee_count_not_negative", "RANGE", "employee_count",
         "employee_count IS NOT NULL AND employee_count >= 0",
         "employee_count is missing or negative"),
    ],
    "suppliers": [
        ("contact_name_not_null", "NOT_NULL", "contact_name",
         "contact_name IS NOT NULL", "contact_name is missing"),
        ("contact_email_valid", "FORMAT", "contact_email",
         "contact_email IS NULL OR contact_email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'",
         "contact_email is invalid"),
        ("lead_time_days_not_negative", "RANGE", "lead_time_days",
         "lead_time_days IS NULL OR lead_time_days >= 0",
         "lead_time_days is negative"),
        ("min_order_value_not_negative", "RANGE", "min_order_value",
         "min_order_value IS NULL OR min_order_value >= 0",
         "min_order_value is negative"),
    ],
    "supplier_products": [
        ("supplier_sku_not_null", "NOT_NULL", "supplier_sku",
         "supplier_sku IS NOT NULL", "supplier_sku is missing"),
        ("supplier_price_positive", "RANGE", "supplier_price",
         "supplier_price IS NULL OR supplier_price > 0",
         "supplier_price is missing or non-positive"),
        ("min_order_qty_at_least_one", "RANGE", "min_order_qty",
         "min_order_qty IS NULL OR min_order_qty >= 1",
         "min_order_qty is below 1"),
        ("lead_time_days_not_negative", "RANGE", "lead_time_days",
         "lead_time_days IS NULL OR lead_time_days >= 0",
         "lead_time_days is negative"),
    ],
}

# referential warn rules: the pipeline predicate uses an internal join flag
# that never lands in silver, so they are evaluated as anti-joins here.
# (rule, column checked, reference table, reference column, reject_reason)
REF_RULES = {
    "logs": [
        ("auth_user_in_customers", "auth_user", "customers", "email",
         "authenticated request's auth_user does not resolve to a current silver customer"),
    ],
}

QUAR_TABLES = {
    "leads": "b2b_ecommerce.quarantine.leads_silver",
    "logs": "b2b_ecommerce.quarantine.logs_silver",
}

# drop-level rule detail for quarantine rejects: rule -> (rule_type, failed_column, reject_reason)
QUAR_RULES = {
    "leads": {
        "lead_id_not_null": ("NOT_NULL", "lead_id", "merge key lead_id is missing - row cannot be merged"),
        "lead_id_format_valid": ("FORMAT", "lead_id", "lead_id does not match LD-########-###"),
        "updated_at_not_null": ("NOT_NULL", "updated_at", "CDC sequence updated_at is missing"),
        "created_date_not_null": ("PARSE", "created_date", "created_date is missing or unparseable"),
        "created_date_not_future": ("DATE_LOGIC", "created_date", "created_date is in the future"),
        "source_valid": ("ENUM", "source", "source is not one of the 7 canonical channels"),
        "status_valid": ("ENUM", "status", "status is not one of the 6 canonical values"),
        "country_valid": ("ENUM", "country", "country is not one of the 4 canonical countries"),
        "company_size_valid": ("ENUM", "company_size", "company_size is not one of the 5 buckets"),
        "estimated_order_volume_valid": ("RANGE", "estimated_order_volume", "estimated_order_volume is negative"),
        "last_contacted_after_created": ("DATE_LOGIC", "last_contacted_date", "last_contacted_date is before created_date"),
    },
    "logs": {
        "client_ip_not_null": ("PARSE", "client_ip", "log line is unparsable - client_ip missing"),
        "event_time_not_null": ("PARSE", "event_time", "log line is unparsable - event_time missing"),
        "method_valid": ("PARSE", "method", "log line is unparsable - method missing or invalid"),
        "status_code_valid": ("PARSE", "status_code", "log line is unparsable - status_code missing or invalid"),
    },
}


def _exception_id(run_id, table, rule, rec):
    return hashlib.md5(f"{run_id}|{table}|{rule}|{rec}".encode()).hexdigest()


def run_dq_audit():
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:6]
    rows = []

    def add(table, rec, rule, rtype, col, val, action, reason):
        rows.append((_exception_id(run_id, table, rule, rec), run_id, SILVER_PIPELINE,
                     table, rec, rule, rtype, col, val, action, reason))

    # 1) warn rules re-evaluated on current silver
    for t, rules in WARN_RULES.items():
        sel = ", ".join(KEYS[t])
        cur = " AND __END_AT IS NULL" if t in SCD2 else ""
        for name, rtype, col, pred, reason in rules:
            for r in spark.sql(
                f"SELECT to_json(struct({sel})) rec, CAST({col} AS STRING) v "
                f"FROM {SILVER}.{t} WHERE NOT coalesce(({pred}), false){cur}"
            ).collect():
                add(f"{SILVER}.{t}", r["rec"], name, rtype, col, r["v"], "WARN_ROW_KEPT", reason)

    # 2) referential warn rules (join-flag based in the pipeline - anti-join here)
    for t, refs in REF_RULES.items():
        for name, col, ref_t, ref_col, reason in refs:
            ref = spark.table(f"{SILVER}.{ref_t}")
            if ref_t in SCD2:
                ref = ref.filter("__END_AT IS NULL")
            viol = (spark.table(f"{SILVER}.{t}")
                    .filter(F.col(col).isNotNull())
                    .join(ref.select(F.col(ref_col).alias("_ref")), F.col(col) == F.col("_ref"), "left_anti")
                    .select(F.to_json(F.struct(*[F.col(k) for k in KEYS[t]])).alias("rec"),
                            F.col(col).cast("string").alias("v")))
            for r in viol.collect():
                add(f"{SILVER}.{t}", r["rec"], name, "REFERENTIAL", col, r["v"], "WARN_ROW_KEPT", reason)

    # 3) drop-level rejects from the quarantine tables (one row per failed rule)
    for t, q in QUAR_TABLES.items():
        rules = QUAR_RULES[t]
        cols = sorted({col for (_, col, _) in rules.values()})
        qdf = spark.table(q).select(
            F.to_json(F.struct(*[F.col(k) for k in KEYS[t]])).alias("rec"),
            "failed_checks", *cols)
        for r in qdf.collect():
            for rule in (r["failed_checks"] or "").split(", "):
                m = rules.get(rule)
                if not m:
                    continue
                rtype, col, reason = m
                add(f"{SILVER}.{t}", r["rec"], rule, rtype, col,
                    (None if r[col] is None else str(r[col])), "DROP_QUARANTINED", reason)

    # write (delete-first makes re-running the same execution idempotent)
    spark.sql(f"DELETE FROM {DQ_TABLE} WHERE audit_run_id = '{run_id}'")
    if rows:
        schema = ("exception_id string, audit_run_id string, pipeline_name string, table_name string, "
                  "source_record string, rule_name string, rule_type string, failed_column string, "
                  "failed_value string, exception_action string, reject_reason string")
        spark.createDataFrame(rows, schema).write.mode("append").saveAsTable(DQ_TABLE)

    summary = (spark.table(DQ_TABLE).filter(F.col("audit_run_id") == run_id)
               .groupBy("table_name", "rule_name", "rule_type", "exception_action")
               .count().orderBy("table_name", "rule_name"))
    tables_hit = len({r[3] for r in rows})
    print(f"DQ exception run {run_id}: {len(rows)} row-level exceptions across {tables_hit} tables")
    return run_id, summary


In [0]:
# Bootstrap + run. Re-runnable: the engine deletes-then-reinserts its own run_id.
spark.sql("""CREATE TABLE IF NOT EXISTS b2b_ecommerce.audit.data_quality_audit (
  exception_id     STRING COMMENT 'Deterministic id: md5(audit_run_id + table + rule + source_record)',
  audit_run_id     STRING COMMENT 'Audit execution that wrote this row',
  pipeline_name    STRING COMMENT 'Pipeline owning the table (attribution across layers)',
  table_name       STRING COMMENT 'Full 3-part table the exception was found in',
  source_record    STRING COMMENT 'Identifying key(s) of the failed row, e.g. {"lead_id":"LD-20260212-002"}',
  rule_name        STRING COMMENT 'Expectation that failed, e.g. created_date_not_future',
  rule_type        STRING COMMENT 'NOT_NULL | FORMAT | RANGE | ENUM | REFERENTIAL | DATE_LOGIC | PARSE',
  failed_column    STRING COMMENT 'Column(s) the rule judged',
  failed_value     STRING COMMENT 'Actual offending value (NULL = value missing/unparsable)',
  exception_action STRING COMMENT 'WARN_ROW_KEPT | DROP_QUARANTINED',
  reject_reason    STRING COMMENT 'Human-readable why this row was rejected/flagged'
) COMMENT 'Row-level DQ exceptions: one row per failed record x failed rule (quarantine rejects + warn re-evaluation)'""")

run_id, summary = run_dq_audit()
display(summary)
